# A1 gate: per-image gain tuning of ADM's one-step denoiser on Urban100 (oracles, R2R, transfer probe)

Can adapting a frozen ADM to one image beat the unadapted one-step posterior mean? Trainable parameters: one gain per
output channel of every convolution plus every GroupNorm γ/β (the rest frozen). Each run: 60 steps of 4 random 256²
crops, Muon (all parameters here are 1-D, so Muon's AdamW branch updates them) at lr 1e-3, fp16 autocast, gradient
checkpointing. Modes, on Urban100 images 8, 16, 24, 32 (the first four of dev-12) at σ = 25:

* oracle@σ′ (the gate): clean crops + fresh noise at σ′ = σ√(1+α²), target = clean, trained at t(σ′); this is the
  expected loss R2R estimates. Applied at t(σ).
* oracle@σ (the ceiling): the same at σ, trained and applied at t(σ).
* R2R (self-supervised): noisy crops y + ασz as input at t(σ′), target y − σz/α, α = 0.5.

Every tuned model is scored on the fixed noisy test image (the seeds of `urban100-one-step`) against the unadapted
model in the same precision; the transfer probe applies parameters tuned on image i to image j.
Pre-registered gate (decision of 2026-11-06, amendment 1): oracle@σ′ mean gain ≥ +0.10 dB and ≥ 3 of 4 images ≥ +0.05,
measured in 512-px-tile mode (the stronger baseline; whole-image mode is also reported), else A1/A2 are killed.
ADM attention uses 4-D q, k, v so the fused SDPA kernels apply at full resolution. Kaggle: GPU T4 ×2, internet on.

In [ ]:
import io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from PIL import Image

SIGMA, ALPHA = 25, 0.5
DEV4 = [8, 16, 24, 32]
STEPS, LR, CROP, BATCH = 60, 1e-3, 256, 4
MODES = ["oracle@σ′", "oracle@σ", "R2R"]
EVAL_MODES = ["whole", "tiles"]   # tiles: 512 px, overlap 128 (the stronger baseline; amendment 1)
SEED_SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)

MUON_SRC = r'''"""Muon: momentum orthogonalised by Newton–Schulz, for hidden weight matrices.

Muon (Jordan et al., 2024) replaces the update of each weight matrix with the
nearest semi-orthogonal matrix to its Nesterov momentum, computed by five
quintic Newton–Schulz iterations; convolution kernels are flattened to
``(out, in · kh · kw)``. It is meant for hidden matrices only, so embeddings,
the output layer, biases and normalisation gains are updated by AdamW inside
the same optimizer, as in the reference implementation.

Muon updates are scaled by ``0.2 · sqrt(max(rows, columns))`` (Liu et al.,
2025, "Muon is Scalable for LLM Training"), which matches their RMS to AdamW's:
learning rates and weight decay tuned for AdamW carry over unchanged.
"""

from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn

NEWTON_SCHULZ = (3.4445, -4.7750, 2.0315)


def newton_schulz(matrix: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Approximate the orthogonal factor ``U V^T`` of ``matrix = U S V^T``; singular values end up in about [0.7, 1.2]."""
    a, b, c = NEWTON_SCHULZ
    native_bf16 = matrix.is_cuda and torch.cuda.get_device_capability(matrix.device)[0] >= 8
    x = matrix.to(torch.bfloat16 if native_bf16 else torch.float32)
    tall = x.size(-2) > x.size(-1)
    if tall:
        x = x.mT
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        gram = x @ x.mT
        x = a * x + (b * gram + c * gram @ gram) @ x
    return (x.mT if tall else x).to(matrix.dtype)


def muon_parameter_groups(model: nn.Module, adamw_modules: Iterable[str] = ("head", "lm_head")) -> list[dict]:
    """Split ``model``'s parameters into a Muon group and an AdamW group.

    AdamW gets every parameter with fewer than two dimensions, every
    ``nn.Embedding`` weight (with any output layer tied to it) and every
    parameter of a module named in ``adamw_modules``, matched against the
    qualified name or its last component. Muon gets the remaining matrices.
    """
    names = set(adamw_modules)
    adamw = {
        id(p)
        for name, module in model.named_modules()
        if isinstance(module, nn.Embedding) or name in names or name.rsplit(".", 1)[-1] in names
        for p in module.parameters(recurse=False)
    }
    muon, other = [], []
    for p in model.parameters():  # tied weights appear once
        (other if p.ndim < 2 or id(p) in adamw else muon).append(p)
    return [{"params": muon, "use_muon": True}, {"params": other, "use_muon": False}]


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices, AdamW for every other parameter.

    ``params`` holds tensors, split by shape (two or more dimensions go to
    Muon), or groups with a ``use_muon`` flag such as
    :func:`muon_parameter_groups` returns. ``lr`` and decoupled
    ``weight_decay`` apply to both kinds; ``momentum``, ``nesterov`` and
    ``ns_steps`` to Muon; ``betas`` and ``eps`` to AdamW.
    """

    def __init__(
        self,
        params,
        lr: float = 3e-4,
        weight_decay: float = 0.0,
        momentum: float = 0.95,
        nesterov: bool = True,
        ns_steps: int = 5,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
    ):
        params = list(params)
        if params and not isinstance(params[0], dict):
            params = [
                {"params": [p for p in params if p.ndim >= 2], "use_muon": True},
                {"params": [p for p in params if p.ndim < 2], "use_muon": False},
            ]
        groups = [{**group, "params": list(group["params"])} for group in params]
        if any("use_muon" not in group for group in groups):
            raise ValueError("every parameter group needs a 'use_muon' flag")
        defaults = dict(
            lr=lr, weight_decay=weight_decay, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, betas=betas, eps=eps,
        )
        super().__init__([group for group in groups if group["params"]], defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            update = self._muon_update if group["use_muon"] else self._adamw_update
            for p in group["params"]:
                if p.grad is None:
                    continue
                direction = update(p.grad, self.state[p], group)
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(direction, alpha=-group["lr"])
        return loss

    @staticmethod
    def _muon_update(grad, state, group):
        if not state:
            state["momentum_buffer"] = torch.zeros_like(grad)
        buffer = state["momentum_buffer"]
        buffer.lerp_(grad, 1 - group["momentum"])
        momentum = grad.lerp(buffer, group["momentum"]) if group["nesterov"] else buffer
        matrix = newton_schulz(momentum.reshape(len(momentum), -1), group["ns_steps"])
        return matrix.reshape_as(grad) * (0.2 * math.sqrt(max(matrix.shape)))

    @staticmethod
    def _adamw_update(grad, state, group):
        beta1, beta2 = group["betas"]
        if not state:
            state["step"] = 0
            state["exp_avg"] = torch.zeros_like(grad)
            state["exp_avg_sq"] = torch.zeros_like(grad)
        state["step"] += 1
        state["exp_avg"].lerp_(grad, 1 - beta1)
        state["exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        corrected = (state["exp_avg_sq"] / (1 - beta2 ** state["step"])).sqrt().add_(group["eps"])
        return state["exp_avg"] / (1 - beta1 ** state["step"]) / corrected
'''  # laboratory/optimization/muon.py
_ns = {"__name__": "muon"}
exec(compile(MUON_SRC, "muon.py", "exec"), _ns)
Muon = _ns["Muon"]


def download(url, path, size=None):
    """Resumable download; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


t0 = time.time()
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
}
with ThreadPoolExecutor(2) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
with zipfile.ZipFile(P["datasets"]) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")], key=natkey)
    TEST = {i: (names[i].split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(names[i]))).convert("RGB"))) for i in DEV4}
TIMES["downloads"] = time.time() - t0
print({i: (n, im.shape) for i, (n, im) in TEST.items()}, f"downloads {TIMES['downloads'] / 60:.1f} min")


def noisy_of(clean_u8, sigma, seed):
    """DMID's protocol, seeds of urban100-one-step: sample std exactly σ/255, unclipped."""
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def psnr(out, clean_u8):
    a = (out[0].clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(clean_u8).permute(2, 0, 1).to(a.device).double()
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

## 1 · ADM in fp32 weights with fp16 autocast and gradient checkpointing; per-channel conv gains + GroupNorm affine

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def checkpoint(func, inputs, params, flag):
    """torch.utils.checkpoint instead of ADM's own (which recomputes outside autocast)."""
    if flag and torch.is_grad_enabled():
        return torch.utils.checkpoint.checkpoint(func, *inputs, use_reentrant=False)
    return func(*inputs)


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(2, 3).reshape(bs, -1, length)


gd_unet.checkpoint = checkpoint
gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_ctx(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=False,
                     use_scale_shift_norm=True, use_checkpoint=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False)
    gains = []
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            g = nn.Parameter(torch.ones(mod.out_channels, device=device))
            mod.register_forward_hook(lambda module, inp, out, g=g: out * g.view(1, -1, 1, 1).to(out.dtype))
            gains.append(g)
    norms = [p for mod in m.modules() if isinstance(mod, nn.GroupNorm) for p in (mod.weight, mod.bias)]
    for p in norms:
        p.requires_grad_(True)
    return dict(dev=device, adm=m, gains=gains, norms=norms, norms0=[p.detach().clone() for p in norms])


CTX = [make_ctx(d) for d in DEVICES]
ADM_STATE.clear()
print("trainable parameters per model:", sum(p.numel() for p in CTX[0]["gains"] + CTX[0]["norms"]))


def get_theta(ctx):
    return [p.detach().clone() for p in ctx["gains"] + ctx["norms"]]


@torch.no_grad()
def set_theta(ctx, theta=None):
    """theta=None restores the unadapted model (gains 1, original GroupNorm affine)."""
    params = ctx["gains"] + ctx["norms"]
    if theta is None:
        theta = [torch.ones_like(g) for g in ctx["gains"]] + ctx["norms0"]
    for p, v in zip(params, theta):
        p.copy_(v)

## 2 · One-step posterior mean (differentiable), tuning loop, full-image evaluation

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def tweedie(model, y, sigma01):
    """One-step posterior mean of y [B,3,H,W] ([0,1] units) at the step matched to sigma01, under fp16 autocast."""
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    with torch.autocast("cuda", dtype=torch.float16):
        eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3]
    return (((x - math.sqrt(1 - ab) * eps.float()) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def crops(img, gen, n=BATCH, size=CROP):
    """n random size² crops of img [3,H,W]."""
    _, H, W = img.shape
    ii = torch.randint(0, H - size + 1, (n,), generator=gen).tolist()
    jj = torch.randint(0, W - size + 1, (n,), generator=gen).tolist()
    return torch.stack([img[:, i:i + size, j:j + size] for i, j in zip(ii, jj)])


def tune(ctx, idx, mode):
    dev, s = ctx["dev"], SIGMA / 255
    name, clean_u8 = TEST[idx]
    clean = torch.from_numpy(clean_u8).permute(2, 0, 1).float().div(255).to(dev)
    noisy = noisy_of(clean_u8, SIGMA, noise_seed(idx)).to(dev)
    set_theta(ctx)
    opt = Muon(ctx["gains"] + ctx["norms"], lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    gen = torch.Generator().manual_seed(1000 * DEV4.index(idx) + MODES.index(mode))
    ngen = torch.Generator(device=dev).manual_seed(77_000 + 1000 * DEV4.index(idx) + MODES.index(mode))
    losses, s_prime = [], s * math.sqrt(1 + ALPHA ** 2)
    for _ in range(STEPS):
        if mode == "R2R":
            c = crops(noisy, gen)
            z = torch.randn(c.shape, generator=ngen, device=dev)
            inp, target, level = c + ALPHA * s * z, c - s * z / ALPHA, s_prime
        else:
            c = crops(clean, gen)
            level = s_prime if mode == "oracle@σ′" else s
            inp, target = c + level * torch.randn(c.shape, generator=ngen, device=dev), c
        loss = F.mse_loss(tweedie(ctx["adm"], inp, level), target)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(loss.item())
    return get_theta(ctx), losses


def noise_seed(idx):
    return 1_000_000 * (SEED_SETS.index("Urban100") + 1) + 1000 * SIGMA + idx


def tiled(fn, y, tile=512, overlap=128):
    """fn applied to overlapping tiles of y, blended with linear ramps (as in urban100-tiles)."""
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    for i in range(0, yp.shape[-2] - tile + 1, stride):
        for j in range(0, yp.shape[-1] - tile + 1, stride):
            acc[..., i:i + tile, j:j + tile] += fn(yp[..., i:i + tile, j:j + tile]) * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


@torch.no_grad()
def evaluate(ctx, idx, theta, how):
    """PSNR of the one-step estimate at t(σ) on the fixed noisy test image, on the whole image or 512-px tiles, with
    parameters theta (None = unadapted)."""
    set_theta(ctx, theta)
    _, clean_u8 = TEST[idx]
    y = noisy_of(clean_u8, SIGMA, noise_seed(idx))[None].to(ctx["dev"])
    one = lambda z: tweedie(ctx["adm"], z, SIGMA / 255)
    out = one(y) if how == "whole" else tiled(one, y)
    set_theta(ctx)
    return psnr(out, clean_u8)

## 3 · Tune every (image, mode); score it on its own image and on the other three

In [ ]:
def run_job(ctx, job):
    idx, mode = job
    t0 = time.time()
    theta, losses = tune(ctx, idx, mode)
    seconds = time.time() - t0
    rows = [dict(tuned_on=idx, evaluated_on=j, mode=mode, eval=how, psnr=evaluate(ctx, j, theta, how), seconds_tuning=seconds)
            for j in DEV4 for how in EVAL_MODES]
    return rows, dict(idx=idx, mode=mode, losses=losses, peak_gib=torch.cuda.max_memory_allocated(ctx["dev"]) / 2 ** 30)


def worker_pool(jobs, fn):
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while True:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads):
        time.sleep(10)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
BASE = {}
for i, idx in enumerate(DEV4):
    for how in EVAL_MODES:
        BASE[(idx, how)] = evaluate(CTX[i % len(CTX)], idx, None, how)
print("unadapted PSNR:", {k: round(v, 3) for k, v in BASE.items()})
res, fail = worker_pool([(idx, mode) for mode in MODES for idx in DEV4], run_job)
TIMES["tuning"] = time.time() - t0
ROWS = pd.DataFrame([r for rows, _ in res for r in rows])
ROWS["base"] = [BASE[(j, how)] for j, how in zip(ROWS.evaluated_on, ROWS["eval"])]
ROWS["gain"] = ROWS.psnr - ROWS.base
ROWS.to_csv(f"{OUT}/gains.csv", index=False)
LOGS = [log for _, log in res]
json.dump(LOGS, open(f"{OUT}/losses.json", "w"))
print(f"{len(res)} runs, {len(fail)} failed, {TIMES['tuning'] / 60:.1f} min; peak GiB:", round(max(l["peak_gib"] for l in LOGS), 2))

## 4 · Gate

In [ ]:
own = ROWS[ROWS.tuned_on == ROWS.evaluated_on].pivot_table(index="evaluated_on", columns=["eval", "mode"], values="gain")
print("gain on the image it was tuned on (dB)")
print(own.round(4).to_string())
cross = ROWS[ROWS.tuned_on != ROWS.evaluated_on].groupby(["eval", "mode"]).gain.mean()
print("\nmean gain when applied to another image (transfer probe):", cross.round(4).to_dict())
gate = own[("tiles", "oracle@σ′")]   # amendment 1: the decision uses the stronger (tiled) baseline
passed = bool(gate.mean() >= 0.10 and (gate >= 0.05).sum() >= 3)
print(f"\ngate oracle@σ′ (tiles): mean {gate.mean():+.4f} dB, {(gate >= 0.05).sum()}/4 images ≥ +0.05 → {'PASS' if passed else 'KILL A1/A2'}")
print("first/last training loss per run:", [(l["idx"], l["mode"], round(l["losses"][0], 6), round(l["losses"][-1], 6)) for l in LOGS])
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, base={f"{i} {how}": v for (i, how), v in BASE.items()},
                   own={f"{how} {mode}": col.to_dict() for (how, mode), col in own.items()},
                   transfer={f"{how} {mode}": v for (how, mode), v in cross.items()}, gate_passed=passed,
                   failed=[j for j, _ in fail]), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))